# VoltIQ – 02: Data Validation

**Objective**: Validate every dataset against its schema — checking required columns, data types, duplicates, null values, numeric ranges, and categorical consistency.

Validation findings are logged and summarised in a quality report.

## 1. Setup & Imports

In [ ]:
import sys, os, logging
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.WARNING,
    format='[%(levelname)s] %(name)s — %(message)s')

from app.utils.data_loader import DataLoader
from app.utils.validation import DataValidator, print_validation_report

loader = DataLoader()
dfs = loader.load_all()
print(f'Loaded {len(dfs)} datasets for validation.')

## 2. Run Full Validation

In [ ]:
validator = DataValidator(dataframes=dfs)
reports = validator.validate_all()

## 3. Print Validation Report

In [ ]:
print_validation_report(reports)

## 4. Pass / Fail Summary

In [ ]:
import pandas as pd

summary = []
for key, r in reports.items():
    summary.append({
        'Dataset':        key,
        'Rows':           r['row_count'],
        'Passed':         r['passed'],
        'Errors':         len(r['errors']),
        'Warnings':       len(r['warnings']),
        'Null Cols':      len(r['null_summary']),
        'Dupe Rows':      r['duplicate_rows'],
        'Range Violations': len(r['range_violations']),
    })

summary_df = pd.DataFrame(summary).set_index('Dataset')
print(summary_df.to_string())

## 5. Null Value Deep Dive

In [ ]:
print('\nNull value breakdown per dataset:')
for key, r in reports.items():
    if r['null_summary']:
        print(f'\n  {key}:')
        for col, n in r['null_summary'].items():
            pct = n / max(r['row_count'], 1) * 100
            print(f'    {col}: {n:,} nulls ({pct:.1f}%)')

## 6. Range Violation Inspection

In [ ]:
print('\nNumeric range violations per dataset:')
for key, r in reports.items():
    if r['range_violations']:
        print(f'\n  {key}:')
        for col, n in r['range_violations'].items():
            print(f'    {col}: {n:,} values out of range')

## 7. Validate a Single Dataset Manually

In [ ]:
# Example: validate only the battery dataset
result = validator.validate('battery', dfs['battery'])
print('Battery validation passed:', result['passed'])
print('Errors:', result['errors'])
print('Warnings:', result['warnings'][:5])

## Summary

Validation complete. All schema checks, range validations, and null analyses are documented above.

**Next step** → `03_Data_Cleaning.ipynb`